In [1]:
import pandas as pd
import numpy as np
from scipy.stats import poisson

# ====================================
# 1. SETUP
# ====================================

try:
    # Load the Stats File
    df_stats = pd.read_excel('football_data.xlsx')
    
    # Clean up text
    df_stats.columns = df_stats.columns.str.strip()
    df_stats['Team'] = df_stats['Team'].astype(str).str.strip()

    # Calculate League Averages
    league_avgs = df_stats.groupby('League')[['Home_xG', 'Home_xGA', 'Away_xG', 'Away_xGA']].mean()
    print(f"Successfully loaded stats for {len(df_stats)} teams from football_data.xlsx.")

except FileNotFoundError:
    print("Error: Could not find 'football_data.xlsx'.")
    exit()
except Exception as e:
    print(f"Error reading Excel file: {e}")
    exit()

def get_match_prediction(home_team, away_team):
    try:
        # 1. GET STATS
        h_stats = df_stats[df_stats['Team'] == home_team].iloc[0]
        a_stats = df_stats[df_stats['Team'] == away_team].iloc[0]
        l_avg = league_avgs.loc[h_stats['League']]
        
        # 2. CALCULATE EFFICIENCY
        # Formula: Season Goals / Season xG
        # We handle potential divide-by-zero by defaulting to 1.0
        if h_stats['Season_xG'] > 0:
            h_eff = h_stats['Season_Goals'] / h_stats['Season_xG']
        else:
            h_eff = 1.0
            
        if a_stats['Season_xG'] > 0:
            a_eff = a_stats['Season_Goals'] / a_stats['Season_xG']
        else:
            a_eff = 1.0
            
        # Keep efficiency between 0.85 (-15%) and 1.15 (+15%)
        # This prevents a lucky team from breaking the model
        h_eff = max(0.85, min(h_eff, 1.15))
        a_eff = max(0.85, min(a_eff, 1.15))
        
        # 3. CALCULATE RAW EXPECTED GOALS (The Engine)
        raw_exp_h = (h_stats['Home_xG']/l_avg['Home_xG']) * (a_stats['Away_xGA']/l_avg['Home_xG']) * l_avg['Home_xG']
        raw_exp_a = (a_stats['Away_xG']/l_avg['Away_xG']) * (h_stats['Home_xGA']/l_avg['Away_xG']) * l_avg['Away_xG']
        
        # 4. APPLY EFFICIENCY ADJUSTMENT
        exp_h = raw_exp_h * h_eff
        exp_a = raw_exp_a * a_eff
        
        # 5. POISSON PROBABILITIES
        max_g = 10
        grid = np.outer(poisson.pmf(np.arange(max_g), exp_h), poisson.pmf(np.arange(max_g), exp_a))
        
        p_away = np.sum(np.triu(grid, 1))
        p_draw = np.sum(np.diag(grid))
        p_home = np.sum(np.tril(grid, -1))
        
        print(f"\n--- {home_team} vs {away_team} ---")
        print(f"Finishing Adj: {home_team} ({h_eff:.2f}x) | {away_team} ({a_eff:.2f}x)")
        print(f"Probabilities: Home {p_home:.1%} | Draw {p_draw:.1%} | Away {p_away:.1%}")
        
        # 6. THE LOGIC
        
        # A. Winner Pick
        if p_home > 0.50:
            print(f" Pick: {home_team} to Win (Strong Confidence)")
        elif p_away > 0.50:
            print(f" Pick: {away_team} to Win (Strong Confidence)")
            
        # B. Draw Detector
        elif p_draw > 0.26:
            print(f" ALERT: High Draw Chance ({p_draw:.1%}). Consider betting DRAW or Double Chance.")
        
        # C. "Value" Trap
        else:
            print(f" Pick: {home_team if p_home > p_away else away_team} (Low Confidence - Risky)")

    except IndexError:
        print(f"Error: Could not find stats for '{home_team}' or '{away_team}'. Check spelling!")
    except KeyError as e:
        print(f"Error: Data missing for one of the teams. Details: {e}")

# ==========================================
# 3. ENTER GAMES HERE
# ==========================================

get_match_prediction("Arsenal", "Chelsea")
get_match_prediction("Fiorentina", "Pisa")

Successfully loaded stats for 96 teams from football_data.xlsx.

--- Arsenal vs Chelsea ---
Finishing Adj: Arsenal (0.99x) | Chelsea (0.85x)
Probabilities: Home 57.4% | Draw 23.4% | Away 19.3%
 Pick: Arsenal to Win (Strong Confidence)

--- Fiorentina vs Pisa ---
Finishing Adj: Fiorentina (0.85x) | Pisa (0.85x)
Probabilities: Home 57.9% | Draw 20.5% | Away 21.7%
 Pick: Fiorentina to Win (Strong Confidence)


In [6]:
import pandas as pd
import numpy as np
from scipy.stats import poisson

# ===================================
# 1. SETUP 
# ===================================
try:
    # Load the Stats File
    df_stats = pd.read_excel('football_data.xlsx') 
    
    # Clean up text
    df_stats.columns = df_stats.columns.str.strip()
    df_stats['Team'] = df_stats['Team'].astype(str).str.strip()
    
    # Calculate League Averages
    league_avgs = df_stats.groupby('League')[['Home_xG', 'Home_xGA', 'Away_xG', 'Away_xGA']].mean()
    print(f"Successfully loaded stats for {len(df_stats)} teams from football_data.xlsx.")
    
except FileNotFoundError:
    print("Error: Could not find 'football_data.xlsx'.")
    exit()
except Exception as e:
    print(f"Error reading Excel file: {e}")
    exit()

def get_match_prediction(home_team, away_team):
    try:
        # 1. GET STATS
        h_stats = df_stats[df_stats['Team'] == home_team].iloc[0]
        a_stats = df_stats[df_stats['Team'] == away_team].iloc[0]
        l_avg = league_avgs.loc[h_stats['League']]
        
        # 2. CALCULATE EFFICIENCY
        if h_stats['Season_xG'] > 0:
            h_eff = h_stats['Season_Goals'] / h_stats['Season_xG']
        else:
            h_eff = 1.0
            
        if a_stats['Season_xG'] > 0:
            a_eff = a_stats['Season_Goals'] / a_stats['Season_xG']
        else:
            a_eff = 1.0
            
        # CLAMP: Keep efficiency between 0.85 (-15%) and 1.15 (+15%)
        h_eff = max(0.85, min(h_eff, 1.15))
        a_eff = max(0.85, min(a_eff, 1.15))
        
        # 3. CALCULATE RAW EXPECTED GOALS
        raw_exp_h = (h_stats['Home_xG']/l_avg['Home_xG']) * (a_stats['Away_xGA']/l_avg['Home_xG']) * l_avg['Home_xG']
        raw_exp_a = (a_stats['Away_xG']/l_avg['Away_xG']) * (h_stats['Home_xGA']/l_avg['Away_xG']) * l_avg['Away_xG']
        
        # 4. APPLY EFFICIENCY ADJUSTMENT
        exp_h = raw_exp_h * h_eff
        exp_a = raw_exp_a * a_eff
        
        # 5. POISSON PROBABILITIES
        max_g = 10
        grid = np.outer(poisson.pmf(np.arange(max_g), exp_h), poisson.pmf(np.arange(max_g), exp_a))
        
        p_away = np.sum(np.triu(grid, 1))
        p_draw = np.sum(np.diag(grid))
        p_home = np.sum(np.tril(grid, -1))

        # --- FIND MOST LIKELY SCORELINE ---
        # np.argmax finds the highest value in the 1D flattened array
        # np.unravel_index converts that back into the 2D grid coordinates (Home Goals, Away Goals)
        most_likely_idx = np.unravel_index(np.argmax(grid), grid.shape)
        pred_h_goals, pred_a_goals = most_likely_idx
        score_prob = grid[pred_h_goals, pred_a_goals]
        
        # --- PRINT BLOCK ---
        print(f"\n--- {home_team} vs {away_team} ---")
        print(f"Finishing Adj: {home_team} ({h_eff:.2f}x) | {away_team} ({a_eff:.2f}x)")
        print(f"Expected Goals (xG): {home_team} {exp_h:.2f} | {away_team} {exp_a:.2f}")
        print(f"Most Likely Score: {home_team} {pred_h_goals} - {pred_a_goals} {away_team} ({score_prob:.1%} chance)")
        print(f"Probabilities: Home {p_home:.1%} | Draw {p_draw:.1%} | Away {p_away:.1%}")
        
        # 6. THE LOGIC
        
        # A. Winner Pick
        if p_home > 0.50:
            print(f"Pick: {home_team} to Win (Strong Confidence)")
        elif p_away > 0.50:
            print(f"Pick: {away_team} to Win (Strong Confidence)")
            
        # B. Draw Detector
        elif p_draw > 0.26:
            print(f"ALERT: High Draw Chance ({p_draw:.1%}). Consider betting DRAW or Double Chance.")
        
        # C. "Value" Trap
        else:
            print(f"Pick: {home_team if p_home > p_away else away_team} (Low Confidence - Risky)")

    except IndexError:
        print(f"Error: Could not find stats for '{home_team}' or '{away_team}'. Check spelling!")
    except KeyError as e:
        print(f"Error: Data missing for one of the teams. Details: {e}")

# ==========================================
# 3. ENTER GAMES HERE
# ==========================================

get_match_prediction("Fulham", "Tottenham")
get_match_prediction("Brighton", "Nottingham")
get_match_prediction("Real Betis", "Sevilla")
get_match_prediction("Hamburger", "Leipzig")

Successfully loaded stats for 96 teams from football_data.xlsx.

--- Fulham vs Tottenham ---
Finishing Adj: Fulham (1.13x) | Tottenham (1.15x)
Expected Goals (xG): Fulham 1.39 | Tottenham 1.31
Most Likely Score: Fulham 1 - 1 Tottenham (12.2% chance)
Probabilities: Home 38.8% | Draw 25.8% | Away 35.4%
 Pick: Fulham (Low Confidence - Risky)

--- Brighton vs Nottingham ---
Finishing Adj: Brighton (0.90x) | Nottingham (0.85x)
Expected Goals (xG): Brighton 1.46 | Nottingham 0.79
Most Likely Score: Brighton 1 - 0 Nottingham (15.4% chance)
Probabilities: Home 53.2% | Draw 26.7% | Away 20.2%
 Pick: Brighton to Win (Strong Confidence)

--- Real Betis vs Sevilla ---
Finishing Adj: Real Betis (0.93x) | Sevilla (1.15x)
Expected Goals (xG): Real Betis 2.12 | Sevilla 1.19
Most Likely Score: Real Betis 2 - 1 Sevilla (9.8% chance)
Probabilities: Home 58.7% | Draw 20.6% | Away 20.7%
 Pick: Real Betis to Win (Strong Confidence)

--- Hamburger vs Leipzig ---
Finishing Adj: Hamburger (0.85x) | Leipzig (0.